# RNN (순환 신경망)
- 이전의 시점의 정보를 현재 시점에 계산에 반영하는 구조

- 첫번째 입력이 들어왔을때
    - 초기의 상태 (h_0)와 입력에 대한 상태 (h_1)로 계산
- 두번째 입력이 들어왔을때
    - 이전의 상태 (h_1)과 결합해서 계산 (h_2)
- 반복 작업이 완료가 되었을때
    - 마지막 상태 (h_T) 생성

- 매개변수
    - input_size
        - 각 시점에서 입력의 피쳐의 개수
    - hidden_size
        - 은닉층(출력)의 크기
    - num_layers
        - 기본값 : 1
        - RNN 층의 개수 (층이 깊을수록 복잡한 패턴이 생성)
        - 개수가 늘어날수록 시간이 증가하고 과적합의 위험성이 존재
        - 1~3 정도 사용을 할때 1부터 개수를 1씩 늘려가면서 사용
    - nonlinearity
        - 기본값 : 'tanh'
        - 비선형 활성화 함수를 선택 
    - bias
        - 기본값 : True
        - 각 가중치에 편향 향을 추가 여부 
    - batch_first
        - 기본값 : False
        - 입력 텐서 첫 차원이 배치인지 여부
    - dropout
        - 기본값 : 0.0
        - 층 사이에 드랍아웃의 적용 비율 -> 층이 2개 이상일 때 사용
    - bidirectional
        - 기본값 : False
        - 양방향 RNN을 사용할지 여부 (순방향 기본, 역방향을 사용할지 지정)

- 매개변수 유의점
    - hidden_size
        - 너무 적은 경우라면 정보가 부족, 너무 큰 경우에는 과적합
        - 일반적으로는 32 ~ 128
    - num_layers
        - 시간의 복잡도에 따라 1 ~ 3 정도 사용
        - 1부터 테스트를 돌려서 교차 검증
    - nolinearity
        - 'tanh'인 기본값이 'relu'로 변경하게 되면 시간은 감소할 수 있지만 불안정
    - dropout
        - 층이 2개인 경우에 사용
        - 과적합 방지를 위해 0.2 ~ 0.5 사용(권장)


In [29]:
# RNN 기본형
# 샘플 데이터를 생성해서 코드 작성
import math
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

In [30]:
# 랜덤 일관성 설정
np.random.seed(42)
torch.manual_seed(42)

In [31]:
# 사인 그래프를 생성 + 노이즈

idx = torch.arange(3000).float()
data = torch.sin(2 * math.pi * 0.02 * idx) + 0.05 * torch.rand(3000)

In [32]:
# 연속성이 있는 시계열데이터를 기준으로 train, test를 8:2의 비율로 나눠준다. 
# 앞부분 80% , 뒷부분 20%
# 전체 데이터의 길이에서 0.8을 곱한 idx의 값이 데이터의 경계
split_idx = int(len(data) * 0.8)
train_data = data[ : split_idx]         # 학습 데이터(구간)
test_data = data[split_idx : ]          # 평가 데이터(구간) 

In [33]:
# train와 test를 스케일링 
scaler = StandardScaler()

train_data = scaler.fit_transform(train_data.reshape(-1, 1))
test_data = scaler.transform(test_data.reshape(-1, 1))

In [34]:
# data들을 torch에서 사용하기 위한 tensor의 형태로 변환 (2차원의 데이터를 1차원의 데이터로 변환)
train_data = torch.tensor(train_data.squeeze(-1), dtype=torch.float32)
test_data = torch.tensor(test_data.squeeze(-1), dtype=torch.float32)

In [35]:
test_data

tensor([-0.0267,  0.1567,  0.3373,  0.5418,  0.6927,  0.8173,  0.9919,  1.0671,
         1.1645,  1.2865,  1.3246,  1.3983,  1.4404,  1.3938,  1.3636,  1.3574,
         1.2469,  1.2248,  1.0723,  0.9846,  0.8028,  0.6572,  0.5244,  0.3643,
         0.2049,  0.0300, -0.1786, -0.3554, -0.5137, -0.6883, -0.8287, -1.0001,
        -1.1170, -1.1681, -1.2985, -1.3153, -1.3834, -1.4009, -1.3774, -1.4069,
        -1.3487, -1.2715, -1.1803, -1.1224, -0.9540, -0.8507, -0.6754, -0.4956,
        -0.3544, -0.1777, -0.0021,  0.2071,  0.3719,  0.5265,  0.6599,  0.8117,
         0.9957,  1.1145,  1.2138,  1.2513,  1.3716,  1.3848,  1.4365,  1.4411,
         1.3921,  1.3294,  1.3120,  1.1749,  1.1121,  0.9337,  0.8198,  0.6503,
         0.5047,  0.3668,  0.1522, -0.0301, -0.1513, -0.3320, -0.5200, -0.7050,
        -0.8393, -0.9781, -1.0663, -1.2039, -1.2880, -1.3682, -1.4039, -1.3903,
        -1.3867, -1.3821, -1.3564, -1.2439, -1.1854, -1.1017, -0.9788, -0.8348,
        -0.6634, -0.5066, -0.3252, -0.17

In [36]:
class WindowDataset(Dataset):
    # 단변량 시계열에서 입력 값 정답값을 만드는 Dataset
    def __init__(self, _data, _window):
        # _data : (N, ) 형태의 1차원 tensor 데이터
        # _window : 과거의 몇개의 데이터를 볼것인가?(구간 설정)
        self.data = _data
        self.window = _window
        # 유효 샘플의 개수 학습 데이터의 개수는 data의 전체 길이에서 -1
        # 입력 데이터는 전체 길이 - 윈도우
        self.n = len(_data) - _window

    # __len__(), __getitem__() 두 개의 특수 함수들은
    # DataLoader에서 자동으로 호출해서 사용이 되는 부분
    def __len__(self):
        return max(self.n,0)
    
    def __getitem__(self, idx):
        # 변환
        # x -> 입력 데이터 (윈도우의 구간을 나타내는 데이터)
        # y -> 입력 데이터 다음 행의 데이터 --> 정답
        x = self.data[idx : idx + self.window ].unsqueeze(-1) # (window, ) -> (window, 1)
        y = self.data[idx + self.window].unsqueeze(-1)
        return x,y
    

In [37]:
# class 생성
# 구간 설정 값
window = 50
train_ds = WindowDataset(train_data, window)
val_ds = WindowDataset(test_data, window)

In [38]:
# DataLoader : 학습 데이터를 구간 별로 뽑아서 새로운 리스트 형 데이터를 생성
train_dl = DataLoader(train_ds, shuffle=True, drop_last=True, batch_size= 64)
val_dl = DataLoader(val_ds, shuffle=False, drop_last=False, batch_size=256)

### DataLoader
- 파이토치에서 Dataset들을 효율적으로 배치 단위로 꺼내주는 반복자
- 반복 학습 루프에 맞는 형태의 데이터셋을 공급을 해주는 역할
- 배치 생성 -> Dataset들을 모아서 텐서의 형태로 묶어 줌
- shuffle -> 에폭마다 데이터의 순서를 랜덤하게 변경
- drop_last -> window에 맞게 구간을 나누고 마지막 구간의 데이터의 개수가 작은 경우 해당 배치를 제거
- pin_memory -> GPU 사용 시 True로 변경 : CPU -> GPU 전송이 빨라짐


In [39]:
class RNNReg(nn.Module):
    # 해당 class 에서 정의되는 함수는 생성자함수, forward() 함수
    def __init__(
        self,
        input_size,
        hidden_size,
        num_layers = 1,
        nonlinearity = 'tanh',
        dropout = 0.0,
        bidirectional = False,
        batch_first = True
        # 매개변수를 입력한 이유 : 
    ):
        # 부모 클래스의 생성자 함수 호출
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size= hidden_size,
            num_layers= num_layers,
            nonlinearity= nonlinearity,
            dropout= dropout,
            bidirectional= bidirectional,
            batch_first= batch_first
        )
        self.head = nn.Linear(hidden_size, 1) # 1차원 스칼라 회귀
        # bidirectional의 값이 False인 경우에는 hidden_size를 사용
        # 만약 True라면 hidden_size * 2
        out_features = hidden_size * (2 if bidirectional else 1)
        self.head = nn.Linear(out_features, 1)
    def forward (self,x):
        # out -> 모든 시점에서의 은닉층의 값(결과)
        # h_n -> 마지막 시점에서의 은닉층의 값(결과) -> 층이 존재하면 층별 마지막 값 ->시계열 분석은 마지막 시점을 사용
        out, h_n = self.rnn(x)
        # 가장 마지막 은늑의 값을 사용(마지막 층의 값)
        last_hidden = h_n[-1]
        res = self.head(last_hidden)
        return res

In [40]:
rnn_model = RNNReg(
    input_size=1, hidden_size=54
)
rnn_model

RNNReg(
  (rnn): RNN(1, 54, batch_first=True)
  (head): Linear(in_features=54, out_features=1, bias=True)
)

In [41]:
# 회귀 분석 -> 손실 함수 -> MSE
criterion = nn.MSELoss()
# 옵티마이저 설정
optimizer = torch.optim.Adam(rnn_model.parameters(), lr= 0.001)
# loss의 미분 폭수 방지용 설정
max_grad_norm = 1.0

# 검증 단계 함수 생성
def evaliate_mse(dataloader):
    rnn_model.eval()
    total_loss = 0.0
    total_n = 0
    with torch.no_grad():
        for x, y in dataloader:
            x = x.float()
            y = y.float()
            yhat = rnn_model(x)
            loss = criterion(yhat, y)
            # 총 손실
            total_loss += loss.item() * y.size(0)
            # loss.item() : 손실의 평균
            # y.size(0) : 개수
            total_n += y.size(0)
    return total_loss / max(total_n, 1)



In [42]:
# 반복 학습 루프

# 학습 데이터의 loss 들 검증 데이터의 loss 들을 저장 -> 시각화
train_history = []
val_history = []

for epoch in range(20):
    rnn_model.train()
    running = 0.0
    n_seen = 0

    for x, y in train_dl:
        x = x.float()
        y = y.float()

        optimizer.zero_grad()
        yhat = rnn_model(x)
        loss = criterion(yhat, y)
        loss.backward()
        # 미분값 폭수 방지
        nn.utils.clip_grad_norm_(rnn_model.parameters(), max_grad_norm)

        optimizer.step()

        running += loss.item() * y.size(0)
        n_seen += y.size(0)
    train_mse = running / max(n_seen, 1)
    val_mse = evaliate_mse(val_dl)
    train_history.append(train_mse)
    val_history.append(val_mse)
    print(f"Epoch : {epoch + 1}, train_mse : {round(train_mse, 7)}, \
          val_mse : {round(val_mse, 3)}")

Epoch : 1, train_mse : 0.2913112,           val_mse : 0.016
Epoch : 2, train_mse : 0.0060675,           val_mse : 0.004
Epoch : 3, train_mse : 0.0028792,           val_mse : 0.002
Epoch : 4, train_mse : 0.002223,           val_mse : 0.002
Epoch : 5, train_mse : 0.0019127,           val_mse : 0.002
Epoch : 6, train_mse : 0.0016765,           val_mse : 0.002
Epoch : 7, train_mse : 0.0016075,           val_mse : 0.001
Epoch : 8, train_mse : 0.0015011,           val_mse : 0.001
Epoch : 9, train_mse : 0.0014192,           val_mse : 0.002
Epoch : 10, train_mse : 0.0014179,           val_mse : 0.001
Epoch : 11, train_mse : 0.0012994,           val_mse : 0.001
Epoch : 12, train_mse : 0.0013568,           val_mse : 0.001
Epoch : 13, train_mse : 0.0013186,           val_mse : 0.001
Epoch : 14, train_mse : 0.0011305,           val_mse : 0.001
Epoch : 15, train_mse : 0.0011286,           val_mse : 0.001
Epoch : 16, train_mse : 0.0010287,           val_mse : 0.001
Epoch : 17, train_mse : 0.0010366,